# Radionica 4, dio 1: naša prva konvoluciona mreža
**AI Lab · GirlTHing**

Danas učimo računar da prepozna odjeću na slici. Koristimo **Fashion-MNIST**: 70.000 malih crno-bijelih slika odjeće u 10 kategorija.

Plan:
1. Pogledamo podatke i vidimo da je slika samo tabela brojeva
2. Napravimo običnu neuronsku mrežu (mnogo logističkih regresija zajedno)
3. Napravimo **CNN** i uporedimo
4. Vidimo gdje griješi (matrica konfuzije)
5. Zavirimo unutra: šta vide filteri?
6. Izazov: ko napravi najbolju mrežu?

### Kako raditi ćelije s TODO
1. Pročitajte tekst **iznad** ćelije: tu piše šta treba uraditi.
2. U ćeliji zamijenite `...` svojim kodom. Ostatak ćelije ne dirajte.
3. Pokrenite ćeliju (Shift + Enter). Ako je sve dobro, nema crvene greške.
4. **Zapele?** Ispod svake TODO ćelije je 💡 **Pomoć**: prvo pročitajte nagovještaj. Rješenje je skriveno ispod njega, otvorite ga tek ako baš ne ide.
5. **Crvena greška?** Pročitajte zadnji red poruke: obično kaže tačno šta fali (npr. zarez ili zagrada).

**Prije početka:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

keras.utils.set_random_seed(42)
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'nema, radimo na CPU (sporije, ali radi)')

## 1. Učitamo podatke
Fashion-MNIST dolazi ugrađen u Keras. Već je podijeljen na train (60.000) i test (10.000).

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

klase = ['majica', 'pantalone', 'džemper', 'haljina', 'kaput',
         'sandala', 'košulja', 'patika', 'torba', 'gležnjača']

print('Train:', X_train.shape, ' Test:', X_test.shape)

**Pitanje:** šta znači oblik `(60000, 28, 28)`? Koliko je to "kolona" po jednoj slici?

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_train[i], cmap='gray_r')
    plt.title(klase[y_train[i]], fontsize=8)
    plt.axis('off')
plt.show()

## 2. Slika je tabela brojeva
Za računar slika nije slika. Pogledajmo jednu sliku kao brojeve.

In [ ]:
slika = X_train[0]
# TODO: ispišite oblik slike, najmanju i najveću vrijednost piksela
print('Oblik:', ...)
print('Najmanji piksel:', ..., ' Najveći piksel:', ...)

# middle part of the image as numbers
print(slika[8:20, 8:20])

> 💡 **Pomoć:** slika je NumPy niz: oblik dobijete sa `.shape`, a najmanju i najveću vrijednost sa `.min()` i `.max()`.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
print('Oblik:', slika.shape)
print('Najmanji piksel:', slika.min(), ' Najveći piksel:', slika.max())
```
</details>

Pikseli idu od 0 (bijelo) do 255 (crno). Mreže bolje uče kad su brojevi mali, pa sve dijelimo s 255.

Dodajemo i jednu dimenziju na kraj: **kanal**. Slike u boji imaju 3 kanala (crveni, zeleni, plavi), kao u CNN Explaineru. Naše imaju samo jedan.

In [ ]:
X_train = (X_train / 255.0).astype('float32')[..., np.newaxis]
X_test = (X_test / 255.0).astype('float32')[..., np.newaxis]
print(X_train.shape)   # (60000, 28, 28, 1)

## 3. Prva mreža: obična (dense) mreža
Sjetite se logističke regresije iz radionice 3: težinska suma plus sigmoid. To je **jedan neuron**.

Obična neuronska mreža je mnogo takvih neurona u slojevima:
- `Flatten` razvuče sliku 28×28 u red od 784 broja
- `Dense(128)` je 128 neurona, svaki gleda svih 784 piksela
- `Dense(10, softmax)` je izlaz: jedan neuron po klasi, vjerovatnoće se sabiraju u 100 %

In [ ]:
dense_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    # TODO: dodajte skriveni sloj sa 128 neurona i aktivacijom 'relu'
    ...,
    # TODO: dodajte izlazni sloj sa 10 neurona i aktivacijom 'softmax'
    ...,
])
dense_model.summary()

> 💡 **Pomoć:** sloj neurona u Kerasu je `layers.Dense(broj_neurona, activation='...')`. Skriveni sloj ima 128 neurona i `'relu'`, izlazni 10 neurona i `'softmax'`. Ne zaboravite zarez na kraju svakog reda u listi.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
layers.Dense(128, activation='relu'),
layers.Dense(10, activation='softmax'),
```
</details>

**Pitanje:** prvi Dense sloj ima 100.480 parametara. Odakle taj broj? (Pomoć: 784 ulaza, 128 neurona, i svaki neuron ima još jedan slobodni član.)

Sad treniramo. `fit` je isti kao u scikit-learnu, samo što mreža prolazi kroz podatke više puta (**epohe**). 10 % train podataka odvajamo za validaciju, kao u radionici 3.

In [ ]:
dense_model.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

dense_hist = dense_model.fit(X_train, y_train, epochs=5,
                             batch_size=64, validation_split=0.1)

## 4. Konvoluciona mreža (CNN)
Dense mreža gleda svaki piksel posebno i ne zna koji pikseli su susjedi. CNN to zna: mali filteri 3×3 klize po slici i traže uzorke, kao u CNN Explaineru.

Recept je isti kao TinyVGG, samo kraći:
**konvolucija → ReLU → pooling**, dva puta, pa izlaz.

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation='relu'),
    layers.MaxPooling2D(pool_size=2),
    # TODO: drugi blok: Conv2D sa 64 filtera 3x3 i 'relu', pa MaxPooling2D(2)
    ...,
    ...,
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
cnn.summary()

> 💡 **Pomoć:** pogledajte prvi blok, dvije linije iznad TODO-a. Drugi blok je isti, samo sa 64 filtera umjesto 32.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
layers.Conv2D(64, kernel_size=3, activation='relu'),
layers.MaxPooling2D(pool_size=2),
```
</details>

**Pogledajte kolonu Output Shape** i uporedite s CNN Explainerom:
- 28×28 → **26×26** nakon konvolucije (filter 3×3 ne može stati na rub)
- 26×26 → **13×13** nakon poolinga (pola)
- 13×13 → **11×11** → **5×5**
- zadnja dimenzija je broj filtera: 32, pa 64

**Pitanje:** prvi Conv2D ima samo 320 parametara, a prvi Dense sloj 100.480. Zašto je razlika tolika?

In [ ]:
# TODO: kompajlirajte i trenirajte CNN isto kao dense mrežu iznad (5 epoha)
cnn.compile(...)

cnn_hist = cnn.fit(...)

> 💡 **Pomoć:** kopirajte `compile` i `fit` iz ćelije gdje smo trenirale dense mrežu (korak 3). Zamijenite `dense_model` sa `cnn`, a `dense_hist` sa `cnn_hist`.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
cnn.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

cnn_hist = cnn.fit(X_train, y_train, epochs=5,
                   batch_size=64, validation_split=0.1)
```
</details>

## 5. Ko je bolji?
Krive učenja: plava je train, narandžasta validacija. Ako train ide gore, a validacija stoji ili pada, mreža uči napamet (**overfitting**).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, hist, naslov in [(axes[0], dense_hist, 'Dense mreža'), (axes[1], cnn_hist, 'CNN')]:
    ax.plot(hist.history['accuracy'], label='train')
    ax.plot(hist.history['val_accuracy'], label='validacija')
    ax.set_title(naslov); ax.set_xlabel('epoha'); ax.set_ylabel('accuracy'); ax.legend()
plt.show()

In [ ]:
_, dense_acc = dense_model.evaluate(X_test, y_test, verbose=0)
_, cnn_acc = cnn.evaluate(X_test, y_test, verbose=0)
print(f'Dense mreža na testu: {dense_acc:.1%}')
print(f'CNN na testu:         {cnn_acc:.1%}')

## 6. Gdje CNN griješi?
Matrica konfuzije iz radionice 3, samo sada 10×10. Redovi su istina, kolone su šta model kaže. Dijagonala su tačni odgovori.

In [ ]:
y_pred = cnn.predict(X_test, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=klase).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
plt.show()

**Pitanje:** koje dvije klase se najviše miješaju? Da li biste i vi pogriješile na tim slikama?

In [ ]:
pogresne = np.where(y_pred != y_test)[0][:12]
proba = cnn.predict(X_test[pogresne], verbose=0)

plt.figure(figsize=(12, 5))
for i, idx in enumerate(pogresne):
    plt.subplot(2, 6, i + 1)
    plt.imshow(X_test[idx, :, :, 0], cmap='gray_r')
    plt.title(f'stvarno: {klase[y_test[idx]]}\nmodel: {klase[y_pred[idx]]} ({proba[i].max():.0%})',
              fontsize=8)
    plt.axis('off')
plt.tight_layout(); plt.show()

## 7. Šta vide filteri?
Isto što smo gledale u CNN Explaineru, sada na našoj mreži: provučemo jednu sliku kroz prvi konvolucioni sloj i nacrtamo svih 32 izlaza. Svijetla mjesta su tamo gdje je filter našao svoj uzorak.

In [ ]:
prvi_sloj = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)

idx = 0   # TODO (opcionalno): probajte i druge slike, npr. 1, 2, 15
mape = prvi_sloj.predict(X_test[idx:idx + 1], verbose=0)[0]

plt.figure(figsize=(3, 3))
plt.imshow(X_test[idx, :, :, 0], cmap='gray_r'); plt.title(klase[y_test[idx]]); plt.axis('off')
plt.show()

plt.figure(figsize=(12, 6))
for f in range(32):
    plt.subplot(4, 8, f + 1)
    plt.imshow(mape[:, :, f], cmap='viridis')
    plt.axis('off')
plt.suptitle('32 mape nakon prve konvolucije i ReLU')
plt.show()

## 8. Izazov: najbolja mreža
Napravite svoju verziju i upišite rezultat na **testu** u zajedničku tabelu.

Šta možete mijenjati:
- broj filtera (`Conv2D(32, ...)` → 64?)
- dodati treći konvolucioni blok
- dodati `layers.Dropout(0.3)` prije zadnjeg Dense sloja (bori se protiv overfittinga)
- broj epoha

**Pravilo:** trening najviše 3 minute. Pobjeđuje najbolji accuracy na testu, a test gledamo samo jednom, na kraju.

In [ ]:
moja_mreza = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    # TODO: vaša arhitektura
    ...,
    layers.Dense(10, activation='softmax'),
])
moja_mreza.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
moja_mreza.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

_, acc = moja_mreza.evaluate(X_test, y_test, verbose=0)
print(f'Moj rezultat na testu: {acc:.1%}')

> 💡 **Pomoć:** krenite od CNN-a iz koraka 4 i promijenite jednu stvar. Korisni dijelovi: `layers.Conv2D(64, 3, activation='relu', padding='same')` (padding='same' čuva veličinu slike), `layers.MaxPooling2D(2)`, `layers.Flatten()`, `layers.Dropout(0.3)`, `layers.Dense(128, activation='relu')`. Ovo je izazov, pa je ispod samo jedan primjer, ne jedino rješenje.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
layers.Conv2D(64, 3, activation='relu', padding='same'),
layers.MaxPooling2D(2),
layers.Conv2D(128, 3, activation='relu', padding='same'),
layers.MaxPooling2D(2),
layers.Flatten(),
layers.Dropout(0.3),
layers.Dense(128, activation='relu'),
```
</details>

---
**Za razmišljanje prije drugog dijela:** naša mreža je učila od nule, na 60.000 slika. Šta ako imamo samo par stotina slika, i to prave fotografije u boji?